In [1]:
import sys
sys.path.append('../')
import numpy as np
import utils_2Q_gate_zp as ut
import qutip as qt

In [2]:
def _parallel_mesolve(n, N, H, tlist, c_op_list, args, options, dims=None):
    col_idx, row_idx = np.unravel_index(n, (N, N))
    rho0 = qt.states.projection(N, row_idx, col_idx)
    rho0.dims = dims
    output = qt.mesolve(
        H, rho0, tlist, c_ops=c_op_list, args=args, options=options,
        _safe_mode=False)
    return output

def get_propagator_noise(H, tlist, num_cpus, parallel, c_op_list, args, options):
    if isinstance(H, list):
        H0 = H[0][0] if isinstance(H[0], list) else H[0]
    else:
        H0 = H
    N = H0.shape[0]
    dims = [H0.dims, H0.dims]
    u = np.zeros([N * N, N * N, len(tlist)], dtype=complex)
    if parallel:
        output = qt.parallel.parallel_map(_parallel_mesolve, range(N * N),
                                task_args=(
                                    N, H, tlist, c_op_list, args, options),
                                task_kwargs={"dims": H0.dims},
                                num_cpus=num_cpus)
        for n in range(N * N):
            for k, t in enumerate(tlist):
                u[:, n, k] = qt.superoperator.mat2vec(output[n].states[k].full()).T
    else:
        for n in range(N * N):
            col_idx, row_idx = np.unravel_index(n, (N, N))
            rho0 = qt.states.projection(N, row_idx, col_idx)
            rho0.dims = H0.dims
            output = qt.mesolve(
                H, rho0, tlist, c_ops=c_op_list, args=args,
                options=options, _safe_mode=False)
            for k, t in enumerate(tlist):
                u[:, n, k] = qt.superoperator.mat2vec(output.states[k].full()).T

    out = np.empty((len(tlist),), dtype=object)
    out[:] = [qt.Qobj(u[:, :, k], dims=dims) for k in range(len(tlist))]
    return out[-1]


In [3]:
def _parallel_mesolve_fast(n, N, H, tlist, c_op_list, args, options, proj_idx, dims=None):
    row_idx, col_idx = proj_idx[n]
    rho0 = qt.states.projection(N, row_idx, col_idx)
    rho0.dims = dims
    output = qt.mesolve(
        H, rho0, tlist, c_ops=c_op_list, args=args, options=options,
        _safe_mode=False)
    return output

def get_propagator_noise_fast(H, tlist, num_cpus, parallel, c_op_list, args, options, logi_state):
    dimz = len(logi_state)
    proj_idx = [(logi_state[i],logi_state[j]) for j in range(dimz)
                for i in range(dimz)]
    if isinstance(H, list):
        H0 = H[0][0] if isinstance(H[0], list) else H[0]
    else:
        H0 = H
    N = H0.shape[0]
    dims = [H0.dims, H0.dims]
    u = np.zeros([N * N, dimz * dimz, len(tlist)], dtype=complex)
    if parallel:
        output = qt.parallel.parallel_map(_parallel_mesolve_fast, range(dimz * dimz),
                                task_args=(
                                    N, H, tlist, c_op_list, args, options, proj_idx),
                                task_kwargs={"dims": H0.dims},
                                num_cpus=num_cpus)
        for n in range(dimz * dimz):
            for k, t in enumerate(tlist):
                u[:, n, k] = qt.superoperator.mat2vec(output[n].states[k].full()).T
        # print((u[:,:,-1]).tolist())
        # print('np.shape(u[:,:,-1])=',np.shape(u[:,:,-1]))
        # print('type(u[:,:,-1])=',type(u[:,:,-1]))
        out = np.empty((len(tlist),), dtype=object)
        out[:] = [qt.Qobj(u[:, :, k], dims=[[[N], [N]], [[dimz], [dimz]]]) for k in range(len(tlist))]
    else:
        for n, idx in enumerate(proj_idx):
            row_idx, col_idx = idx
            rho0 = qt.states.projection(N, row_idx, col_idx)
            # print(N, H0.dims, c_op_list[0].dims, rho0.dims)
            rho0.dims = H0.dims
            output = qt.mesolve(
                H, rho0, tlist, c_ops=c_op_list, args=args,
                options=options, _safe_mode=False)
            for k, t in enumerate(tlist):
                u[:, n, k] = qt.superoperator.mat2vec(output.states[k].full()).T

        out = np.empty((len(tlist),), dtype=object)
        out[:] = [qt.Qobj(u[:, :, k], dims=[[[N], [N]], [[dimz], [dimz]]]) for k in range(len(tlist))]
    return out[-1]

def get_fidelity(s_op, keep_levels):
    p0_kraus = qt.to_kraus(qt.to_super(s_op))
    p0_kraus = [ut.truncate_2(i, keep_levels) for i in p0_kraus]
    p0_super_2 = qt.kraus_to_super(p0_kraus)
    f_noise = qt.metrics.average_gate_fidelity(p0_super_2, target=qt.sigmax())
    return f_noise

In [7]:
# test qt.propagator with fast version for a qutrit
w = 1
logi_state = [0,2]
keep_levels = [0,2]
gamma = 0.01
t_final = np.pi/2
tlist = np.linspace(0, t_final, 10)
zero = qt.basis(3,0)
one = qt.basis(3,1)
two = qt.basis(3,2)

H = [w*(qt.states.projection(3, 0, 2) + qt.states.projection(3, 2, 0))]

c_op_list = [np.sqrt(gamma)*qt.states.projection(3, 2, 0)]
parallel = False
num_cpus = 1
options =qt.Options(max_step=1e-4, nsteps=1e4, num_cpus=30 )
p_simple_a = get_propagator_noise(H, tlist, num_cpus, parallel, c_op_list,
                            args=None, options=options)
f_noise_simple = get_fidelity(p_simple_a, keep_levels)
print('f_noise_2 = ', f_noise_simple)

p_simple_2_a = get_propagator_noise_fast(H, tlist, num_cpus, parallel, c_op_list,
                                   args=None, options=options, logi_state=logi_state)
f_noise_simple_2 = get_fidelity(p_simple_2_a, keep_levels)

print('f_noise_2 = ', f_noise_simple_2)

f_noise_2 =  0.9947921805432897
f_noise_2 =  0.994792180543291


In [5]:
N=2
for n in range(N * N):
    col_idx, row_idx = np.unravel_index(n, (N, N))
    print(n, row_idx, col_idx)

print(' ')
logi_state = [0,2]
dimz = 2
proj_idx = [(logi_state[i],logi_state[j]) for j in range(dimz)
                for i in range(dimz)]
for n, idx in enumerate(proj_idx):
    row_idx, col_idx = idx
    print(n, row_idx, col_idx)

0 0 0
1 1 0
2 0 1
3 1 1
 
0 0 0
1 2 0
2 0 2
3 2 2


In [6]:
# test qt.propagator with fast version for a qubit
w = 1
gamma = 0.001
t_final = np.pi/2
tlist = np.linspace(0, t_final, 10)
H = [w*qt.sigmax()]
c_op_list = [np.sqrt(gamma)*qt.sigmam()]
parallel = True
num_cpus = 2
logi_state = [0,1]
p_simple = get_propagator_noise(H, tlist, num_cpus, parallel, c_op_list,
                            args=None, options=options)
f_noise_simple = qt.metrics.average_gate_fidelity(p_simple, target=qt.sigmax())
print('f_noise_2 = ', f_noise_simple)

p_simple_2 = get_propagator_noise_fast(H, tlist, num_cpus, parallel, c_op_list,
                                   args=None, options=options, logi_state=logi_state)
f_noise_simple_2 = qt.metrics.average_gate_fidelity(p_simple_2, target=qt.sigmax())
print('f_noise_2 = ', f_noise_simple_2)

f_noise_2 =  0.9994766838421407
f_noise_2 =  0.9994766838421407
